# Capítulo 13 — Gêmeos Digitais em Odontologia

Este notebook explora o conceito de gêmeos digitais aplicado à odontologia:
representação computacional do estado bucal do paciente para simulação
de intervenções e prognósticos.

In [ ]:
# ============================================================
# 1. Configuração
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import torch

from odontoia.models.digital_twin import (
    DentalTwin, ToothState, build_twin_encoder, estimate_intervention
)

print("Módulos carregados!")

In [ ]:
# ============================================================
# 2. Criação do gêmeo digital de um paciente
# ============================================================
paciente = DentalTwin(patient_id='PAC-2026-001')

print(f'Paciente: {paciente.patient_id}')
print(f'Dentes: {len(paciente.teeth)}')
print(f'Estágio periodontal: {paciente.periodontal_stage}')
print(f'Índice de higiene: {paciente.hygiene_index:.1f}/5')
print(f'Densidade óssea: {paciente.bone_density:.0f}%')

# Altera alguns dentes para simular condição real
for t in paciente.teeth:
    if t.numero in [16, 17, 26, 27]:  # Molares
        t.saudavel = False
        t.carie = True
        t.profundidade_sondagem = 5.5
    if t.numero == 36:  # Lesão periapical
        t.lesao_periapical = True
        t.profundidade_sondagem = 7.0

print(f'\nDentes comprometidos: {sum(1 for t in paciente.teeth if not t.saudavel)}')

In [ ]:
# ============================================================
# 3. Visualização simplificada da arcada
# ============================================================
def plot_dental_twin(twin, title="Gêmeo Digital - Arcada Dentária"):
    """Plota representação simplificada da arcada."""
    fig, ax = plt.subplots(figsize=(14, 6))
    
    for t in twin.teeth:
        if t.saudavel:
            cor = 'green'
        elif t.carie and not t.restaurado:
            cor = 'red'
        elif t.restaurado:
            cor = 'orange'
        else:
            cor = 'gray'
        
        ax.scatter(t.pos_x, t.pos_y, c=cor, s=200, edgecolors='black', linewidth=1, zorder=5)
        ax.annotate(str(t.numero), (t.pos_x, t.pos_y),
                    ha='center', va='center', fontsize=7, fontweight='bold', color='white')
    
    # Moldura da arcada
    theta = np.linspace(0, np.pi, 100)
    x_sup = 50 * np.cos(theta - np.pi/2)
    y_sup = 30 * np.sin(np.pi - theta) + 60
    x_inf = 50 * np.cos(theta - np.pi/2)
    y_inf = -30 * np.sin(np.pi - theta) - 60
    
    ax.plot(x_sup, y_sup, 'k--', alpha=0.2, linewidth=0.5)
    ax.plot(x_inf, y_inf, 'k--', alpha=0.2, linewidth=0.5)
    
    ax.set_xlim(-65, 65)
    ax.set_ylim(-100, 100)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')
    
    # Legenda
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='green', markersize=12, label='Saudável'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=12, label='Cárie'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='orange', markersize=12, label='Restaurado'),
    ]
    ax.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3)
    
    plt.tight_layout()
    plt.show()

plot_dental_twin(paciente, 'Gêmeo Digital - Antes da Intervenção')

In [ ]:
# ============================================================
# 4. Simulação de intervenções
# ============================================================
# Intervenção 1: Restauração
resultado_rest = estimate_intervention(paciente, {"tipo": "restauracao", "dentes": [16, 17, 26, 27]})
print(f"Restauração: {resultado_rest['intervencao']['dentes']}")
print(f"  Pré: {resultado_rest['diagnostico_pre']['dentes_comprometidos']} dentes comprometidos")

# Intervenção 2: Tratamento periodontal
resultado_perio = estimate_intervention(paciente, {"tipo": "periodontal", "tratamento": "raspagem"})
print(f"\nTratamento periodontal:")
print(f"  Estágio: {resultado_perio['diagnostico_pre']['periodontal_stage']} -> {resultado_perio['prognostico']['periodontal_stage']}")

# Intervenção 3: Higiene oral
resultado_hig = estimate_intervention(paciente, {"tipo": "higiene"})
print(f"\nMelhora da higiene:")
print(f"  Índice: {resultado_hig['diagnostico_pre']['hygiene_index']:.1f} -> {resultado_hig['prognostico']['hygiene_index']:.1f}")

In [ ]:
# ============================================================
# 5. Autoencoder para representação latente
# ============================================================
encoder = build_twin_encoder(latent_dim=16, input_dim=260)

# Converte o paciente para vetor e codifica
vec = paciente.to_feature_vector()
vec_tensor = torch.tensor(vec, dtype=torch.float32).unsqueeze(0)

with torch.no_grad():
    z, recon = encoder(vec_tensor)

print(f'Vetor original: {vec.shape}')
print(f'Espaço latente: {z.shape} -> {z.numpy().flatten()[:8]}...')
print(f'Erro de reconstrução: {((vec_tensor - recon) ** 2).sum().item():.4f}')

# Múltiplos pacientes no espaço latente
pacientes = []
for i in range(10):
    p = DentalTwin(patient_id=f'PAC-{i:04d}')
    pacientes.append(p.to_feature_vector())

X = torch.tensor(np.array(pacientes), dtype=torch.float32)
with torch.no_grad():
    Z, _ = encoder(X)

# Visualização 2D (PCA improvisado: pega 2 primeiras dimensões latentes)
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(Z[:, 0], Z[:, 1], c=range(10), cmap='viridis', s=100)
for i in range(10):
    ax.annotate(f'PAC-{i:04d}', (Z[i, 0].item(), Z[i, 1].item()), fontsize=8)
ax.set_title('Pacientes no Espaço Latente do Gêmeo Digital', fontsize=13)
ax.set_xlabel('Dimensão Latente 1')
ax.set_ylabel('Dimensão Latente 2')
plt.tight_layout()
plt.show()

---
**Conclusão:** Gêmeos digitais permitem simular intervenções e
monitorar a evolução do paciente de forma não-invasiva.